# 01 Boundaries and the service crosswalk

Fetches fire and rescue authority polygons from the Office for National Statistics
feature service, reconciles them against the territories recorded in the incident
panel, and writes `config/service_crosswalk.csv`. It then assigns HadUK-Grid land cells to
territories and writes the label array used by the weather aggregation.

Run after `scripts/convert_raw.py`.

In [1]:
import pandas as pd

from fdh import boundaries as bd
from fdh import contract
from fdh.config import boundary_layer, load_config, repo_root

config = load_config()
raw_dir = config.paths.resolved("raw_dir")
interim_dir = config.paths.resolved("interim_dir")
raw_dir, interim_dir

(WindowsPath('D:/Chris Masters/Final Research Project/Chris-MScfire-demand-horizon/fire-demand-horizon/data/raw'),
 WindowsPath('D:/Chris Masters/Final Research Project/Chris-MScfire-demand-horizon/fire-demand-horizon/data/interim'))

## Fetch

The layer named in `boundaries.layer` is served in EPSG:27700, which matches the HadUK-Grid projection, so no
reprojection is needed before the aggregation.

In [2]:
layer = boundary_layer(config)
collection = bd.fetch_layer(layer.url)
record = bd.save_layer(collection, raw_dir / "boundaries" / layer.filename)
record

{'filename': 'fra_2022_en_bgc.geojson',
 'features': 44,
 'properties': ['BNG_E',
  'BNG_N',
  'FID',
  'FRA22CD',
  'FRA22NM',
  'GlobalID',
  'LAT',
  'LONG',
  'Shape__Area',
  'Shape__Length'],
 'crs': {'type': 'name', 'properties': {'name': 'EPSG:27700'}},
 'sha256': '78faea92daebd13ef80f043a020de75a589c0721509f36e8925fc0f736e7f934',
 'bytes': 5092876}

## Reconcile

The join is on E code first and normalised name second.

In [3]:
polygons, code_field, name_field = bd.load_layer(raw_dir / "boundaries" / layer.filename)
territories, by_year = bd.territories_from_panel(interim_dir)

print(f"{len(polygons)} boundary features ({code_field}, {name_field})")
print(f"{len(territories)} incident territories")

44 boundary features (FRA22CD, FRA22NM)
43 incident territories


In [4]:
rows = bd.build_crosswalk(territories, polygons)
crosswalk = pd.DataFrame(rows)
crosswalk

,e_code_territory,frs_territory,boundary_code,boundary_name,match_basis,name_differs
0,E31000001,Avon,E31000001,Avon,e_code,False
1,E31000002,Bedfordshire,E31000002,Bedfordshire,e_code,False
2,E31000003,Berkshire,E31000003,Royal Berkshire,e_code,True
3,E31000004,Buckinghamshire,E31000004,Buckinghamshire & Milton Keynes,e_code,True
4,E31000005,Cambridgeshire,E31000005,Cambridgeshire,e_code,False
5,E31000006,Cheshire,E31000006,Cheshire,e_code,False
6,E31000007,Cleveland,E31000007,Cleveland,e_code,False
7,E31000008,Cornwall and Isles of Scilly,E31000008,Cornwall,e_code,True
8,E31000009,Cumbria,E31000009,Cumbria,e_code,False
9,E31000010,Derbyshire,E31000010,Derbyshire,e_code,False


## What did not line up

In [5]:
print("unmatched incident territories:")
print(crosswalk.loc[crosswalk.match_basis == "unmatched"].to_string(index=False) or "  none")

print("\nmatched on name, E codes differ:")
print(crosswalk.loc[crosswalk.match_basis == "normalised_name"].to_string(index=False))

print("\nboundary features with no incident territory:")
for code, name in bd.unmatched_boundaries(rows, polygons):
    print(f"  {code}  {name}")

unmatched incident territories:
Empty DataFrame
Columns: [e_code_territory, frs_territory, boundary_code, boundary_name, match_basis, name_differs]
Index: []

matched on name, E codes differ:
e_code_territory               frs_territory boundary_code               boundary_name     match_basis  name_differs
       E31000017 Hampshire and Isle of Wight     E31000048 Hampshire and Isle of Wight normalised_name         False

boundary features with no incident territory:
  E31000039  Isles of Scilly


In [6]:
crosswalk.loc[crosswalk.name_differs, ["e_code_territory", "frs_territory", "boundary_name"]]

,e_code_territory,frs_territory,boundary_name
2,E31000003,Berkshire,Royal Berkshire
3,E31000004,Buckinghamshire,Buckinghamshire & Milton Keynes
7,E31000008,Cornwall and Isles of Scilly,Cornwall
12,E31000013,Durham,County Durham and Darlington
16,E31000046,Greater London,London Fire and Emergency Planning Authority
31,E31000030,Nottinghamshire,Nottinghamshire and City of Nottingham
35,E31000033,Staffordshire,Stoke-on-Trent and Staffordshire


## Coverage by financial year

Authorities entering or leaving the series show here.

In [7]:
counts = {year: len(codes) for year, codes in sorted(by_year.items())}
everything = set().union(*by_year.values())
missing = {
    year: sorted((code, territories[code]) for code in everything - codes)
    for year, codes in sorted(by_year.items())
    if everything - codes
}
counts, missing

({'2010/11': 43,
  '2011/12': 43,
  '2012/13': 43,
  '2013/14': 43,
  '2014/15': 43,
  '2015/16': 43,
  '2016/17': 43,
  '2017/18': 43,
  '2018/19': 43,
  '2019/20': 43,
  '2020/21': 43,
  '2021/22': 43,
  '2022/23': 43,
  '2023/24': 43,
  '2024/25': 43,
  '2025/26': 43},
 {})

## Write the crosswalk

The `note` column is for decisions taken by hand: which polygon a merged
territory uses, and what happens to any boundary feature with no territory.
Notes already in the file are carried over when it is rewritten.

In [8]:
out = repo_root() / "config" / "service_crosswalk.csv"
bd.write_crosswalk(rows, out)
print(f"wrote {out}")

wrote D:\Chris Masters\Final Research Project\Chris-MScfire-demand-horizon\fire-demand-horizon\config\service_crosswalk.csv


## Rural urban classification

The Office for National Statistics Rural Urban Classification 2021 of fire and
rescue areas. It is an attribute table with no geometry covering England and
Wales, so it is fetched as a table rather than a layer and filtered to England.

The join uses FRA25 codes, so Hampshire and Isle of Wight carries E31000048 here
and E31000017 in the incident data, the same mismatch the crosswalk already
handles. The join is therefore on `boundary_code`, not `e_code_territory`. The
boundary layer carries FRA22 codes, so an authority whose code changed between
2022 and 2025 would show below as unclassified rather than be matched by name.

In [9]:
classification_source = config.sources["ons_fra_classification"][0]
table_path = raw_dir / "boundaries" / classification_source.filename
table_rows = bd.fetch_table(classification_source.url)
table_record = bd.save_table(table_rows, table_path)
table_record

{'filename': 'fra_2025_ruc21_ew.json',
 'features': 47,
 'properties': ['FRA25CD',
  'FRA25NM',
  'ObjectId',
  'Proportion_of_population_in_OAs_further_from_a_major_town_or_city____',
  'Proportion_of_population_in_rural_OAs____',
  'RUC21CD',
  'RUC21NM',
  'RUC21_relative_access',
  'RUC21_settlement_class',
  'Rural_Urban_flag'],
 'sha256': 'f449616724b0a0b300fcb695621a9989d81f7dc73b4827c48cccc8b99186474b',
 'bytes': 20026}

In [10]:
classification = bd.load_classification(table_path)
england = {code: row for code, row in classification.items() if code.startswith("E")}
print(f"{len(classification)} classified areas, {len(england)} in England")

47 classified areas, 44 in England


In [11]:
ruc = pd.DataFrame(list(england.values()))
classified = crosswalk.merge(
    ruc, how="left", left_on="boundary_code", right_on=contract.CLASSIFICATION_KEY
).sort_values(contract.RURAL_SHARE, ascending=False)
classified[
    [
        "e_code_territory",
        "frs_territory",
        "boundary_code",
        contract.SETTLEMENT_CLASS,
        contract.RURAL_SHARE,
    ]
]

,e_code_territory,frs_territory,boundary_code,RUC21_settlement_class,Proportion_of_population_in_rural_OAs____
7,E31000008,Cornwall and Isles of Scilly,E31000008,Majority rural,55.4
8,E31000009,Cumbria,E31000009,Majority rural,50.4
27,E31000026,Norfolk,E31000026,Intermediate rural,48.7
25,E31000025,Lincolnshire,E31000025,Intermediate rural,47.5
30,E31000029,Northumberland,E31000029,Intermediate rural,45.1
28,E31000027,North Yorkshire,E31000027,Intermediate rural,42.4
36,E31000034,Suffolk,E31000034,Intermediate rural,40.1
33,E31000032,Shropshire,E31000032,Intermediate rural,38.6
10,E31000011,Devon and Somerset,E31000011,Intermediate rural,37.9
12,E31000013,Durham,E31000013,Intermediate rural,37.9


In [12]:
unclassified = classified.loc[
    classified[contract.CLASSIFICATION_KEY].isna(),
    ["e_code_territory", "frs_territory", "boundary_code", "match_basis"],
]
print("crosswalk rows with no classification:")
print(unclassified.to_string(index=False) if len(unclassified) else "  none")

crosswalk rows with no classification:
  none


## Grid cells per territory

Each HadUK-Grid land cell is assigned to the territory whose polygon contains
its centre. The grid is identical across variables and months (notebook 03), so
the assignment is built once from one reference file and applies to the whole
archive.

The matched boundary code is mapped through the crosswalk, so the result carries
the E code recorded in the incident data. Isles of Scilly has its own polygon
and no recorded territory, so `boundaries.merged` sends its cells to Cornwall
and Isles of Scilly. The polygons come from the layer named in
`boundaries.layer`.

In [13]:
from fdh import grid
from fdh import weather as wx
from fdh.config import require

probe_dir = raw_dir / "weather" / "probe"
held = sorted(probe_dir.glob("*.nc"))
if held:
    reference = held[0]
else:
    variable = require(config.weather.variables, "weather.variables")[0]
    listing = wx.daily_files(
        require(config.weather.base_url, "weather.base_url"),
        variable,
        require(config.weather.version_directory, "weather.version_directory"),
        config.weather.window_start,
        config.weather.release_covers_to,
    )
    reference = wx.download(listing[0], probe_dir)
print(f"reference file: {reference.name}")

reference file: rainfall_hadukgrid_uk_1km_day_20091001-20091031.nc


### Guards

Each of these stops the build when it fails: validity constant within the
reference file, 1000 m spacing in both directions, boundary coordinates in
British National Grid metres, and at least one cell for every territory in the
crosswalk.

In [14]:
fill = require(config.weather.fill_value, "weather.fill_value")
eastings, northings, land = grid.read_grid(reference, fill)
print(f"PASS  validity constant within the file: {int(land.sum()):,} land cells "
      f"of {land.size:,}, grid {land.shape[0]} by {land.shape[1]}")
print(f"PASS  spacing {grid.SPACING_METRES:.0f} m in both directions")

shapes = grid.load_boundaries(raw_dir / "boundaries" / layer.filename)
west, south, east, north = shapes.total_bounds
print(f"PASS  boundary coordinates in metres: eastings {west:,.0f} to {east:,.0f}, "
      f"northings {south:,.0f} to {north:,.0f}, {len(shapes)} polygons from "
      f"{layer.era}, {shapes.crs}")

lookup = grid.load_crosswalk(out, config.boundaries.merged)
cells = grid.land_cells(eastings, northings, land)
codes, assignment = grid.assign(cells, shapes, lookup)
print(f"PASS  every crosswalk territory receives cells: {assignment['territories']} "
      f"of {len(set(lookup.values()))}")

PASS  validity constant within the file: 245,077 land cells of 1,305,000, grid 1450 by 900
PASS  spacing 1000 m in both directions
PASS  boundary coordinates in metres: eastings 82,669 to 655,654, northings 5,353 to 657,539, 44 polygons from fra_2022_en_bgc, EPSG:27700
PASS  every crosswalk territory receives cells: 43 of 43


### Reported, not stopped

HadUK-Grid covers the whole United Kingdom and the boundary layer covers England
only, so most land cells outside every polygon are Scotland, Wales, Northern
Ireland and the Isle of Man. The rest are coastal cells whose centre falls
outside a polygon clipped to mean high water.

In [15]:
print(f"land cells:                     {assignment['land_cells']:,}")
print(f"assigned to a territory:        {assignment['assigned']:,}")
print(f"outside every polygon:          {assignment['outside_every_polygon']:,}")
print(f"boundary codes with no crosswalk entry: {assignment['no_crosswalk_entry'] or 'none'}")

land cells:                     245,077
assigned to a territory:        130,135
outside every polygon:          114,942
boundary codes with no crosswalk entry: none


### Cells per territory

In [16]:
summary = grid.summarise(cells, codes)
per_territory = (
    pd.DataFrame(summary)
    .merge(crosswalk[["e_code_territory", "frs_territory"]], on="e_code_territory")
    .sort_values("cells", ascending=False)
    .reset_index(drop=True)
)
print(per_territory[["e_code_territory", "frs_territory", "cells"]].to_string())

   e_code_territory                 frs_territory  cells
0         E31000011            Devon and Somerset  10122
1         E31000027               North Yorkshire   8312
2         E31000009                       Cumbria   6782
3         E31000047          Dorset and Wiltshire   6128
4         E31000025                  Lincolnshire   5904
5         E31000026                       Norfolk   5367
6         E31000029                Northumberland   5027
7         E31000017   Hampshire and Isle of Wight   4124
8         E31000018        Hereford and Worcester   3915
9         E31000034                       Suffolk   3787
10        E31000022                          Kent   3712
11        E31000015                         Essex   3636
12        E31000008  Cornwall and Isles of Scilly   3530
13        E31000020                    Humberside   3500
14        E31000032                    Shropshire   3491
15        E31000005                Cambridgeshire   3404
16        E31000023            

### Cells against polygon area

A cell is one square kilometre, so the cells assigned to a territory should
come close to the area of its polygons in square kilometres, the Isles of
Scilly polygon counted with Cornwall. The area is computed from the polygons
themselves. A ratio far from one would mean cells assigned to the wrong polygon
or coordinates in other units; a small shortfall is expected on coasts, where a
cell centre can fall outside a polygon clipped to mean high water.

In [17]:
square_km = shapes.geometry.area.groupby(shapes.boundary_code.map(lookup)).sum() / 1e6
areas = per_territory.set_index("e_code_territory")[["frs_territory", "cells"]].assign(
    polygon_km2=square_km.round(0)
)
areas["cells_per_km2"] = (areas.cells / areas.polygon_km2).round(3)
print(areas.sort_values("cells_per_km2").to_string())
print(f"\nall territories: {areas.cells.sum():,} cells against "
      f"{areas.polygon_km2.sum():,.0f} square kilometres")

                                 frs_territory  cells  polygon_km2  cells_per_km2
e_code_territory                                                                 
E31000041                           Merseyside    631        652.0          0.968
E31000008         Cornwall and Isles of Scilly   3530       3565.0          0.990
E31000015                                Essex   3636       3670.0          0.991
E31000022                                 Kent   3712       3739.0          0.993
E31000009                              Cumbria   6782       6823.0          0.994
E31000042                      South Yorkshire   1542       1551.0          0.994
E31000017          Hampshire and Isle of Wight   4124       4148.0          0.994
E31000020                           Humberside   3500       3516.0          0.995
E31000025                         Lincolnshire   5904       5936.0          0.995
E31000007                            Cleveland    595        598.0          0.995
E31000011       

### Write the label array

`grid_labels.npy` holds a service index per grid cell, -1 where no service is
assigned. `grid_labels_index.json` maps each index to its E code and records
what the mask was built from: the boundary layer and the hash of its file, the
hash of the crosswalk, and the merges declared in config. The weather panel
build stops if any of those differs from the current configuration. The
per-territory counts go to the reports directory.

In [18]:
labels, index = grid.label_array(land.shape, cells, codes)
built_from = grid.mask_basis(
    layer.era, raw_dir / "boundaries" / layer.filename, out, config.boundaries.merged
)
array_path, index_path = grid.save_labels(
    labels,
    index,
    eastings,
    northings,
    reference,
    config.paths.resolved("processed_dir"),
    built_from,
)
counts_path = config.paths.resolved("reports_dir") / "grid_cells_per_territory.csv"
grid.write_summary(summary, counts_path)
print(f"wrote {array_path}")
print(f"wrote {index_path}")
print(f"wrote {counts_path}")
built_from

wrote D:\Chris Masters\Final Research Project\Chris-MScfire-demand-horizon\fire-demand-horizon\data\processed\grid_labels.npy
wrote D:\Chris Masters\Final Research Project\Chris-MScfire-demand-horizon\fire-demand-horizon\data\processed\grid_labels_index.json
wrote D:\Chris Masters\Final Research Project\Chris-MScfire-demand-horizon\fire-demand-horizon\reports\grid_cells_per_territory.csv


{'boundaries_layer': 'fra_2022_en_bgc',
 'boundaries_sha256': '78faea92daebd13ef80f043a020de75a589c0721509f36e8925fc0f736e7f934',
 'crosswalk_sha256': 'd511b751bd01b637c8d43b63b396baffe8d17d35d96dcf432cc3ec10427276c7',
 'merged_boundaries': {'E31000039': 'E31000008'}}